In [ ]:
import cv2
import torch
import requests
from time import time
from IPython.display import display, Video

# Step 1: Download Video from URL
video_url = "https://sample-videos.com/video123/mp4/720/big_buck_bunny_720p_1mb.mp4"  # Change to your video URL
video_path = "input_video.mp4"

with open(video_path, "wb") as file:
    file.write(requests.get(video_url).content)

print(f"Downloaded video saved as {video_path}")

# Step 2: Clone YOLOv5 and Install Requirements
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt

In [ ]:
model = torch.hub.load("ultralytics/yolov5", "yolov5s", pretrained=True)
import os
def process_video(video_path, output_path, model):
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    temp_output = "output_raw.mp4"
    out = cv2.VideoWriter(temp_output, fourcc, fps, (width, height))

    frame_count = 0
    start_time = time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = model(frame_rgb)

        for result in results.xyxy[0]:
            x1, y1, x2, y2, conf, cls = result
            label = f"{model.names[int(cls)]}: {conf:.2f}"
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.putText(frame, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        frame_count += 1
        elapsed_time = time() - start_time
        current_fps = frame_count / elapsed_time
        cv2.putText(frame, f"FPS: {current_fps:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        out.write(frame)

    cap.release()
    out.release()
    print(f"Processed video saved to {temp_output}")

    # Convert to H.264 for Colab compatibility
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 {output_path} -y")
    print(f"Final converted video saved as {output_path}")

In [ ]:
from google.colab import files
from IPython.display import display, Video

print("Upload your video now:")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]  # Get uploaded video filename
print(f"Uploaded video: {video_path}")

# Step 5: Process Video
output_video_path = "output.mp4"
process_video(video_path, output_video_path, model)

# Step 6: Display the Processed Video
display(Video(output_video_path, embed=True))